## 基 2 快速傅里叶变换的实现

Import standard modules:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML 
HTML('../style/course.css') #apply general CSS

import cmath

本练习要求实现一个基于 Python 的快速傅里叶变换（FFT）。在 [$\S$ 2.8 &#10142;](2_8_the_discrete_fourier_transform.ipynb) 的基础上，我们将用时间抽取（DIT）方式，为输入长度 $N = 2^n$ 的一维函数实现基 2 Cooley-Tukey FFT，然后再把这个函数推广到任意长度的输入。

根据 [$\S$ 2.8.2 &#10142;](2_8_the_discrete_fourier_transform.ipynb)，离散傅里叶变换（DFT）的定义为：

$$ \mathscr{F}_{\rm D}\{y\}_k = Y_k =  \sum_{n\,=\,0}^{N-1} y_n\,e^{-\imath 2\pi \frac{nk}{N}}, $$

也就是说，傅里叶变换后频谱 $Y$ 的第 $k$ 个元素，是对函数 $y$ 的全部 $n$ 个元素求和，每一项都乘以复旋转因子 $e^{-\imath 2\pi \frac{nk}{N}}$。在 [$\S$ 2.8.5 &#10142;](2_8_the_discrete_fourier_transform.ipynb) 中，我们介绍了针对长度 $N = 2^n$ 的离散函数计算 DFT 的两种方法：一种是用双重循环逐个计算频谱元素，另一种是构造傅里叶核 $K$ 后做矩阵乘法。执行 DFT 的计算复杂度为 $\mathcal{O}(N^2)$，也就是说需要大约 $cN^2$ 次运算，其中 $c > 1$ 为常数因子。不过，正如 [$\S$ 2.8.5 &#10142;](2_8_the_discrete_fourier_transform.ipynb) 所指出的，矩阵实现通常比循环实现快得多，因为它利用了高效的向量化数学库。

下面再次给出 DFT 代码，以便与我们实现的 FFT 进行比较：

In [ ]:
def loop_DFT(x):
    """
    Implementing the DFT in a double loop
    Input: x = the vector we want to find the DFT of
    """
    #Get the length of the vector (will only work for 1D arrays)
    N = x.size
    #Create vector to store result in
    X = np.zeros(N, dtype=complex)
    for k in range(N):
        for n in range(N):
            X[k] += np.exp(-1j * 2.0* np.pi* k * n / N) * x[n]
    return X

def matrix_DFT(x):
    """
    Implementing the DFT in vectorised form
    Input: x = the vector we want to find the DFT of
    """
    #Get the length of the vector (will only work for 1D arrays)
    N = x.size
    #Create vector to store result in
    n = np.arange(N)
    k = n.reshape((N,1))
    K = np.exp(-1j * 2.0 * np.pi * k * n / N)
    return K.dot(x)

在 [$\S$ 2.8.6 &#10142;](2_8_the_discrete_fourier_transform.ipynb) 中，我们已经介绍过快速傅里叶变换：它利用递归，在 $\mathcal{O}(N\log_2N)$ 次运算内完成傅里叶变换，从而显著降低计算成本，尤其是在 $N$ 很大时。那里还给出了一个“单层”快速傅里叶变换的例子：先把输入函数拆成两部分，再在调用基于矩阵的 DFT 之前，对这一层中的所有值施加旋转因子。下面再次给出这段代码。

In [ ]:
def one_layer_FFT(x):
    """An implementation of the 1D Cooley-Tukey FFT using one layer"""
    N = x.size
    if N%2 > 0:
        print "Warning: length of x is not a power of two, returning DFT"
        return matrix_DFT(x)
    else:
        X_even = matrix_DFT(x[::2])
        X_odd = matrix_DFT(x[1::2])
        factor = np.exp(-2j * np.pi * np.arange(N) / N)
        return np.concatenate([X_even + factor[:N / 2] * X_odd, X_even + factor[N / 2:] * X_odd])

我们可以通过构造一个离散测试函数 $x$，并比较各个函数调用的输出，轻松验证这些函数会给出相同的结果：

In [ ]:
xTest = np.random.random(256)  # create random vector to take the DFT of

print np.allclose(loop_DFT(xTest), matrix_DFT(xTest)) # returns True if all values are equal (within numerical error)
print np.allclose(matrix_DFT(xTest), one_layer_FFT(xTest)) # returns True if all values are equal (within numerical error)

我们还可以对每个函数计时，比较生成完整频谱所需的时间。

In [ ]:
print 'Double Loop DFT:'
%timeit loop_DFT(xTest)
print '\nMatrix DFT:'
%timeit matrix_DFT(xTest)
print '\nOne Layer FFT + Matrix DFT:'
%timeit one_layer_FFT(xTest)

可以看到，矩阵形式的 DFT 明显快于双重循环版 DFT，这是因为 `numpy` 的向量化运算非常高效。而“单层”FFT 又大约比矩阵 DFT 快一倍，这是 FFT 架构带来的优势。再进一步，我们还可以使用 `numpy` 内置的 FFT：

In [ ]:
print np.allclose(one_layer_FFT(xTest), np.fft.fft(xTest))

print 'numpy FFT:'
%timeit np.fft.fft(xTest)

`numpy` 的 FFT 非常快，一方面得益于底层程序实现，另一方面更根本的原因是它采用了 FFT 架构。本练习的目标，就是亲自实现这样的架构。

### 时间抽取（DIT）FFT（12 分）

FFT 的计算效率来自算法的递归设计：它利用了二叉树结构以及广义的*旋转因子*。基于二叉树的组织方式不同，可以得到*时间抽取（DIT）*和*频率抽取（DIF）*两种架构。两者产生的结果等价，但在 FFT 二叉树上的计算方向和起始位置不同。关于 DIT 实现的示意图和伪代码，可参阅维基百科中的 [Cooley-Tukey FFT &#10548;](https://en.wikipedia.org/wiki/Cooley%E2%80%93Tukey_FFT_algorithm) 页面。

本节练习要求你实现基 2 DIT FFT 算法，输入长度为 $2^n$，输入既可以是实数也可以是复数。

In [ ]:
def ditrad2(x):
    """radix-2 DIT FFT
    x: list or array of N values to perform FFT on, can be real or imaginary, x must be of size 2^n
    """
    ox = np.asarray(x, dtype='complex') # assure the input is an array of complex values
    # INSERT: assign a value to N, the size of the FFT
    N = #??? 1 point
    
    if N==1: return ox # base case

    # INSERT: compute the 'even' and 'odd' components of the FFT,
    # you will recursively call ditrad() here on a subset of the input values
    # Hint: a binary tree design splits the input in half
    even = #??? 2 points
    odd = #??? 2 points
    
    twiddles = np.exp(-2.j * cmath.pi * np.arange(N) / N) # compute the twiddle factors

    # INSERT: apply the twiddle factors and return the FFT by combining the even and odd values
    # Hint: twiddle factors are only applied to the odd values
    # Hint: combing even and odd is different from the way the inputs were split apart above.
    return #??? 3 points

当 `ditrad2()` 正确实现后，它的输出应与 `numpy` FFT 的结果等价，而且运行速度应快于 DFT 和单层 FFT。

In [ ]:
print 'The output of ditrad2() is correct?', np.allclose(np.fft.fft(xTest), ditrad2(xTest)) # 2 points if true

print 'your FFT:'
%timeit ditrad2(xTest) # 2 point if your time < One Layer FFT + Matrix DFT

### 非 $2^n$ 长度的 FFT（10 分）

既然已经为长度 $2^n$ 的向量实现了快速的基 2 算法，我们现在可以编写一个可处理任意长度输入的通用算法。这个算法需要先检查输入长度是否可被 2 整除；如果可以，就使用 FFT，否则退回到较慢的矩阵 DFT。

In [ ]:
def generalFFT(x):
    """radix-2 DIT FFT
    x: list or array of N values to perform FFT on, can be real or imaginary
    """
    ox = np.asarray(x, dtype='complex') # assure the input is an array of complex values
    # INSERT: assign a value to N, the size of the FFT
    N = #??? 1 point
    
    if N==1: return ox # base case
    elif # INSERT: check if the length is divisible by 2, 1 point

        # INSERT: do a FFT, use your ditrad2() code here, 3 points
        # Hint: your ditrad2() code can be copied here, and will work with only a minor modification
        
    else: # INSERT: if not divisable by 2, do a slow Fourier Transform
        return # ??? 1 point

当我们用该算法处理不同长度的输入时，运行时间应当有所不同。若向量长度是质数，算法将退回到较慢的矩阵 DFT；若向量长度几乎总能不断被 2 整除，那么算法就应当更快。

In [ ]:
xTest2 = np.random.random(251)  # create random vector to take the DFT of, not, this is not of length 2^n
xTest3 = np.random.random(12*32)  # create random vector to take the DFT of, not, this is not of length 2^n

In [ ]:
print 'The output of generalFFT() is correct?', np.allclose(np.fft.fft(xTest2), generalFFT(xTest2)) # 1 point

print 'Your generic FFT:'
%timeit generalFFT(xTest2) # 1 point if it runs in approximately the same time as matrix_DFT

%timeit generalFFT(xTest3) # 2 point if it runs faster than the xTest2 vector

### 后续扩展

* 原地 FFT、定点实现、基 2 DIF、基 4